# 2D Kadane's Algorithm (Maximum Sum Submatrix)

**2D Kadane's Algorithm** extends 1D Kadane's Algorithm to solve the **Maximum Sum Submatrix (Rectangle) Problem**:

> Given an $R \times C$ 2D integer matrix, find the contiguous rectangular submatrix (containing at least one cell) that has the largest sum, and return its sum (and optionally its bounding coordinates).

---

## Core Intuition: Reducing 2D to 1D

Instead of checking all $O(R^2 C^2)$ rectangles independently:

1. **Fix the top and bottom row boundaries** (`top` from `0` to `R - 1`, and `bottom` from `top` to `R - 1`).
2. **Compress the rows `[top .. bottom]` into a single 1D array `col_sums` of size `C`**, where:
   $$\text{col\_sums}[c] = \sum_{r=\text{top}}^{\text{bottom}} \text{matrix}[r][c]$$
   Because `bottom` advances one row at a time for a fixed `top`, we update `col_sums[c] += matrix[bottom][c]` in $O(1)$ per column instead of re-summing from scratch.
3. **Run 1D Kadane's Algorithm on `col_sums`** to find the contiguous column range `[left .. right]` with the maximum sum for that `(top, bottom)` slice.
4. **Update the global maximum** across all `(top, bottom)` pairs.

> **Tip:** If a matrix has many rows and few columns ($R > C$), iterate over `(left, right)` column pairs and compress into a 1D row array instead. That gives $O(\min(R, C)^2 \cdot \max(R, C))$ time!

## Example & Step-by-Step Execution

Consider this $4 \times 5$ matrix:

```python
matrix = [
    [ 1,  2, -1, -4, -20],
    [-8, -3,  4,  2,   1],
    [ 3,  8, 10,  1,   3],
    [-4, -1,  1,  7,  -6],
]
```

### Step-by-Step Trace

Below are the compressed `col_sums` arrays and the 1D Kadane result for each `(top, bottom)` pair:

| `(top, bottom)` | Compressed `col_sums` | 1D Kadane Best Sum | Best `(left, right)` | Global `max_sum` |
| :---: | :--- | :---: | :---: | :---: |
| `(0, 0)` | `[1, 2, -1, -4, -20]` | `3` | `(0, 1)` | `3` |
| `(0, 1)` | `[-7, -1, 3, -2, -19]` | `3` | `(2, 2)` | `3` |
| `(0, 2)` | `[-4, 7, 13, -1, -16]` | `20` | `(1, 2)` | `20` |
| `(0, 3)` | `[-8, 6, 14, 6, -22]` | `26` | `(1, 3)` | `26` |
| `(1, 1)` | `[-8, -3, 4, 2, 1]` | `7` | `(2, 4)` | `26` |
| `(1, 2)` | `[-5, 5, 14, 3, 4]` | `26` | `(1, 4)` | `26` |
| **`(1, 3)`** | **`[-9, 4, 15, 10, -2]`** | **`29`** | **`(1, 3)`** | **`29`** |
| `(2, 2)` | `[3, 8, 10, 1, 3]` | `25` | `(0, 4)` | `29` |
| `(2, 3)` | `[-1, 7, 11, 8, -3]` | `26` | `(1, 3)` | `29` |
| `(3, 3)` | `[-4, -1, 1, 7, -6]` | `8` | `(2, 3)` | `29` |

### How 1D Kadane Finds `29` on `(top=1, bottom=3)`
`col_sums = [-9, 4, 15, 10, -2]`
- `c = 0`: `cur = -9`, `best = -9`
- `c = 1`: `max(4, -9 + 4) = 4` (**Restart** at `left = 1`), `best = 4`
- `c = 2`: `max(15, 4 + 15) = 19` (**Extend**), `best = 19`
- `c = 3`: `max(10, 19 + 10) = 29` (**Extend**), `best = 29` (`left = 1, right = 3`)
- `c = 4`: `max(-2, 29 - 2) = 27` (**Extend**), `best = 29`

### Optimal Submatrix
- **Maximum Sum:** `29`
- **Bounds:** rows `1..3`, columns `1..3`:
  ```text
  [-3,  4,  2]
  [ 8, 10,  1]
  [-1,  1,  7]
  ```

In [3]:
def kadane_1d(arr: list[int]) -> tuple[int, int, int]:
    """Return (max_sum, start_idx, end_idx) for a non-empty 1D array."""
    cur_sum = best_sum = arr[0]
    temp_start = best_start = best_end = 0

    for i in range(1, len(arr)):
        if arr[i] > cur_sum + arr[i]:
            cur_sum = arr[i]
            temp_start = i
        else:
            cur_sum += arr[i]

        if cur_sum > best_sum:
            best_sum = cur_sum
            best_start = temp_start
            best_end = i

    return best_sum, best_start, best_end


def max_submatrix_sum(matrix: list[list[int]], trace: bool = False) -> tuple[int, tuple[int, int, int, int]]:
    """Return (max_sum, (top, left, bottom, right)) using 2D Kadane's Algorithm."""
    rows, cols = len(matrix), len(matrix[0])
    global_max = matrix[0][0]
    best_bounds = (0, 0, 0, 0)

    if trace:
        print(f"{'(top, bot)':<10} | {'col_sums':<25} | {'1D Max':>6} | {'(L, R)':<8} | {'Global Max':>10}")
        print("-" * 72)

    for top in range(rows):
        col_sums = [0] * cols
        for bottom in range(top, rows):
            for c in range(cols):
                col_sums[c] += matrix[bottom][c]

            cur_max, left, right = kadane_1d(col_sums)
            if cur_max > global_max:
                global_max = cur_max
                best_bounds = (top, left, bottom, right)

            if trace:
                print(f"({top}, {bottom})     | {str(col_sums):<25} | {cur_max:>6} | ({left}, {right})   | {global_max:>10}")

    return global_max, best_bounds


def max_submatrix_sum_optimized(matrix: list[list[int]]) -> int:
    """Optimize by compressing along the smaller dimension: O(min(R, C)^2 * max(R, C))."""
    rows, cols = len(matrix), len(matrix[0])
    if rows > cols:
        # Transpose logically by accessing matrix[r][c] with outer loop on columns
        global_max = matrix[0][0]
        for left in range(cols):
            row_sums = [0] * rows
            for right in range(left, cols):
                for r in range(rows):
                    row_sums[r] += matrix[r][right]
                cur_max, _, _ = kadane_1d(row_sums)
                global_max = max(global_max, cur_max)
        return global_max

    best, _ = max_submatrix_sum(matrix)
    return best


# Example run
matrix = [
    [ 1,  2, -1, -4, -20],
    [-8, -3,  4,  2,   1],
    [ 3,  8, 10,  1,   3],
    [-4, -1,  1,  7,  -6],
]

best_sum, (top, left, bottom, right) = max_submatrix_sum(matrix, trace=True)
print(f"\nMax Submatrix Sum: {best_sum}")
print(f"Bounds: top={top}, left={left}, bottom={bottom}, right={right}")
print("Submatrix:")
for r in range(top, bottom + 1):
    print(matrix[r][left:right + 1])

# Edge-case checks
assert best_sum == 29
assert max_submatrix_sum_optimized(matrix) == 29
assert max_submatrix_sum([[-5, -2], [-8, -1]])[0] == -1       # All negative
assert max_submatrix_sum([[7]])[0] == 7                        # Single cell
assert max_submatrix_sum([[2, -1, 3, -2, 4]])[0] == 6          # Single row
assert max_submatrix_sum_optimized([[2], [-1], [3], [-2], [4]]) == 6  # Tall matrix (R > C)

(top, bot) | col_sums                  | 1D Max | (L, R)   | Global Max
------------------------------------------------------------------------
(0, 0)     | [1, 2, -1, -4, -20]       |      3 | (0, 1)   |          3
(0, 1)     | [-7, -1, 3, -2, -19]      |      3 | (2, 2)   |          3
(0, 2)     | [-4, 7, 13, -1, -16]      |     20 | (1, 2)   |         20
(0, 3)     | [-8, 6, 14, 6, -22]       |     26 | (1, 3)   |         26
(1, 1)     | [-8, -3, 4, 2, 1]         |      7 | (2, 4)   |         26
(1, 2)     | [-5, 5, 14, 3, 4]         |     26 | (1, 4)   |         26
(1, 3)     | [-9, 4, 15, 10, -2]       |     29 | (1, 3)   |         29
(2, 2)     | [3, 8, 10, 1, 3]          |     25 | (0, 4)   |         29
(2, 3)     | [-1, 7, 11, 8, -3]        |     26 | (1, 3)   |         29
(3, 3)     | [-4, -1, 1, 7, -6]        |      8 | (2, 3)   |         29

Max Submatrix Sum: 29
Bounds: top=1, left=1, bottom=3, right=3
Submatrix:
[-3, 4, 2]
[8, 10, 1]
[-1, 1, 7]


## Time and Space Complexity

Let $R$ be the number of rows and $C$ be the number of columns.

| Approach | Time Complexity | Space Complexity | Notes |
| :--- | :---: | :---: | :--- |
| Brute Force (all submatrices, recompute sum) | $O(R^3 C^3)$ | $O(1)$ | Checks $O(R^2 C^2)$ rectangles, $O(RC)$ per sum |
| 2D Prefix Sums (all submatrices) | $O(R^2 C^2)$ | $O(RC)$ | $O(1)$ per rectangle sum, still $O(R^2 C^2)$ pairs |
| **2D Kadane's (row compression)** | **$O(R^2 \cdot C)$** | **$O(C)$** | $\frac{R(R+1)}{2}$ row pairs $\times O(C)$ update & Kadane |
| **2D Kadane's (smaller-dimension first)** | **$O(\min(R, C)^2 \cdot \max(R, C))$** | **$O(\max(R, C))$** | Best general complexity; ideal for rectangular matrices |

### Why It Is $O(R^2 \cdot C)$ Time and $O(C)$ Space
- **Outer `(top, bottom)` loops:** There are $\frac{R(R + 1)}{2} = O(R^2)$ pairs of rows.
- **Inner work per pair:** Updating `col_sums` takes $O(C)$ and running 1D Kadane's on `col_sums` takes $O(C)$, giving $O(R^2 \cdot C)$ total time.
- **Space:** We only store one 1D array `col_sums` of length $C$ plus a handful of scalar variables — $O(C)$ extra space (no $O(RC)$ prefix-sum matrix needed).

## Related Problems & Extensions

Pure **Maximum Sum Submatrix** appears directly on GeeksforGeeks / interview platforms, and its **2D-to-1D compression** technique powers several classic LeetCode Hard and Medium problems:

| # | Problem | Difficulty | Connection to 2D Kadane's Algorithm |
| :-: | :--- | :---: | :--- |
| 1 | **363. Max Sum of Rectangle No Larger Than K** | Hard | Compresses 2D matrix into 1D `col_sums` in $O(R^2 C)$; uses 1D Kadane's as an $O(C)$ fast path (`if cur_max <= k`) before falling back to sorted prefix sums (`bisect`). |
| 2 | **1074. Number of Submatrices That Sum to Target** | Hard | Uses the exact same `(top, bottom)` row compression into 1D `col_sums`, then counts subarrays summing to `target` with a prefix-sum hashmap. |
| 3 | **53. Maximum Subarray** | Medium | The 1D Kadane subroutine executed inside every `(top, bottom)` iteration. |
| 4 | **1749. Maximum Absolute Sum of Any Subarray** | Medium | Extends directly to **Maximum Absolute Sum Submatrix** by running both max-Kadane and min-Kadane on `col_sums`. |
| 5 | **85. Maximal Rectangle** | Hard | Compresses a binary 2D matrix row-by-row into 1D column heights, then solves 1D Largest Rectangle in Histogram. |
| 6 | **1727. Largest Submatrix With Rearrangements** | Medium | Row-wise consecutive-1s compression into 1D heights followed by sorting each row. |
| 7 | **1504. Count Submatrices With All Ones** | Medium | Compresses matrix rows into 1D height histograms and counts valid rectangles per row. |
| 8 | **1292. Maximum Side Length of a Square with Sum <= Threshold** | Medium | 2D prefix sum technique for constrained square submatrices. |

In [2]:
import bisect


def max_sum_submatrix_no_larger_than_k(matrix: list[list[int]], k: int) -> int:
    """LeetCode 363: 2D compression + 1D Kadane fast-path + sorted prefix sums."""
    rows, cols = len(matrix), len(matrix[0])
    ans = float("-inf")

    for top in range(rows):
        col_sums = [0] * cols
        for bottom in range(top, rows):
            for c in range(cols):
                col_sums[c] += matrix[bottom][c]

            # Fast path via 1D Kadane's: if max subarray <= k, no binary search needed
            cur = best = col_sums[0]
            for val in col_sums[1:]:
                cur = max(val, cur + val)
                best = max(best, cur)
            if best <= k:
                ans = max(ans, best)
                continue

            # General path: find smallest prefix >= running_sum - k
            prefixes = [0]
            running = 0
            for val in col_sums:
                running += val
                idx = bisect.bisect_left(prefixes, running - k)
                if idx < len(prefixes):
                    ans = max(ans, running - prefixes[idx])
                bisect.insort(prefixes, running)

    return int(ans)


# Sanity checks
assert max_sum_submatrix_no_larger_than_k([[1, 0, 1], [0, -2, 3]], 2) == 2
assert max_sum_submatrix_no_larger_than_k([[2, 2, -1]], 3) == 3
assert max_sum_submatrix_no_larger_than_k([[2, 2, -1]], 0) == -1
print("All 2D variant checks passed!")

All 2D variant checks passed!
